# Rede neural MLP para classificação de digitos com scikit-learn

### Import das bibliotecas

In [ ]:
import matplotlib.pyplot as plt# Para plotar imagens
from sklearn.datasets import fetch_openml# Importar um dataset chamado MNIST
from sklearn import metrics# Acuracia, Precisao, F1-Score e Recall
from sklearn.neural_network import MLPClassifier# RNA MLP
from sklearn.model_selection import train_test_split as tts# Dividir o dataset em treino e teste

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

X = X / 255.0

In [ ]:
X_train, X_test, y_train, y_test = tts(X, y, random_state=42, test_size=0.3)

### Criando a rede neural MLP

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=10, alpha=1e-4,
                    solver="sgd", verbose=10, random_state=42, learning_rate_init=0.2)

### Visulizando os digitos

In [ ]:
from sklearn import datasets
digits = datasets.load_digits()

_, axes = plt.subplots(nrows=1, ncols=4, figsize=(8, 3))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.set_axis_off()
    ax.imshow(image, cmap=plt.cm.gray_r, interpolation="nearest")
    ax.set_title("Training: %i" % label)

### Treinando a rede neural para classificar imagens

In [ ]:
mlp.fit(X_train, y_train)

In [ ]:
print(f"Acuracia da rede neural: {100 * round(mlp.score(X_test, y_test), 4)}%")

### Calculando outras metricas (Precisao, F1-Score, Recall)

In [ ]:
predict_mlp = mlp.predict(X_test)

In [ ]:
print(
    f"Classification report for classifier {mlp}:\n"
    f"{metrics.classification_report(y_test, predict_mlp)}\n"
)

In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_predictions(y_test, predict_mlp)
disp.figure_.suptitle("Confusion Matrix")
print(f"Confusion matrix:\n{disp.confusion_matrix}")

plt.show()

---
# Rede neural MLP para classificação de dígitos com PyTorch

Agora vamos construir **a mesma rede** do exemplo anterior, mas com PyTorch: mesma base (MNIST), mesma divisão treino/teste e mesma arquitetura (784 → 10 → 10 → 10 → 10).

| | scikit-learn | PyTorch |
|---|---|---|
| Modelo | `MLPClassifier(...)` pronto | Montamos as camadas uma a uma |
| Treinamento | `mlp.fit()` faz tudo | Escrevemos o laço: *forward* → perda → *backward* → atualização |
| Controle | Menor, porém mais simples | Total sobre cada etapa |
| GPU | Não usa | Usa, se disponível |

### Import das bibliotecas

In [ ]:
import numpy as np
import torch
import torch.nn as nn# camadas e funções de perda
from torch.utils.data import TensorDataset, DataLoader# organiza os dados em lotes (batches)

torch.manual_seed(42)# Semente para resultados reprodutíveis
device = "cuda" if torch.cuda.is_available() else "cpu"# usa a GPU do Colab, se estiver ativada
print("Dispositivo:", device)

### Preparando os dados

Reaproveitamos `X_train`, `X_test`, `y_train` e `y_test` do exemplo anterior. O PyTorch trabalha com **tensores**: as imagens viram `float32` e os rótulos (que vêm como texto `'0'`...`'9'`) viram inteiros.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train.astype(int))# rótulos de texto -> inteiros (classe 0 a 9)
y_test_t = torch.tensor(y_test.astype(int))

# Lotes de 200 imagens, mesmo tamanho padrão usado pelo MLPClassifier
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=200, shuffle=True)
print("Treino:", tuple(X_train_t.shape), "| Teste:", tuple(X_test_t.shape))

### Criando a rede neural MLP

Cada `nn.Linear` é uma camada totalmente conectada (cada neurônio faz Σ wᵢxᵢ + b) e `nn.ReLU` é a função de ativação, a mesma usada por padrão no `MLPClassifier`.

In [ ]:
modelo = nn.Sequential(
    nn.Linear(784, 10), nn.ReLU(),# 1ª camada oculta: 784 pixels -> 10 neurônios
    nn.Linear(10, 10), nn.ReLU(),# 2ª camada oculta
    nn.Linear(10, 10), nn.ReLU(),# 3ª camada oculta
    nn.Linear(10, 10),# camada de saída: 1 neurônio por dígito (0 a 9)
).to(device)

for camada in modelo:
    if isinstance(camada, nn.Linear):# inicialização de Glorot/Xavier, a mesma do scikit-learn
        nn.init.xavier_uniform_(camada.weight)
        nn.init.uniform_(camada.bias, -np.sqrt(6 / sum(camada.weight.shape)), np.sqrt(6 / sum(camada.weight.shape)))
print(modelo)
print("Total de parâmetros:", sum(p.numel() for p in modelo.parameters()))

### Função de perda e otimizador

Usamos os mesmos hiperparâmetros do exemplo com scikit-learn: SGD com taxa de aprendizado 0.2, *momentum* de Nesterov 0.9 e regularização L2 (`alpha=1e-4` vira `weight_decay`).

In [ ]:
criterio = nn.CrossEntropyLoss()# aplica softmax e calcula a entropia cruzada
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.2, momentum=0.9,
                             nesterov=True, weight_decay=1e-4)
EPOCAS = 10# equivale ao max_iter=10

### Treinando a rede neural

Em cada lote acontecem os 4 passos do treinamento: **forward** (previsão), **perda**, **backward** (retropropagação do gradiente) e **atualização** dos pesos.

In [ ]:
historico = []
for epoca in range(EPOCAS):
    modelo.train()# modo de treino
    perda_total = 0.0
    for imagens, rotulos in train_loader:
        imagens, rotulos = imagens.to(device), rotulos.to(device)
        saidas = modelo(imagens)# 1. forward: calcula as saídas da rede
        perda = criterio(saidas, rotulos)# 2. compara as saídas com os rótulos
        otimizador.zero_grad()# zera os gradientes do lote anterior
        perda.backward()# 3. backward: calcula os gradientes
        otimizador.step()# 4. atualiza os pesos
        perda_total += perda.item() * len(imagens)
    historico.append(perda_total / len(train_loader.dataset))
    print(f"Época {epoca + 1:2d}/{EPOCAS} - perda = {historico[-1]:.4f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCAS + 1), historico, marker="x")
plt.xlabel("Época"); plt.ylabel("Perda (entropia cruzada)"); plt.title("Curva de aprendizado")
plt.grid(alpha=0.3); plt.show()

### Avaliando no conjunto de teste

In [ ]:
modelo.eval()                                              # modo de avaliação
with torch.no_grad():                                      # desliga o cálculo de gradientes na inferência
    saidas_teste = modelo(X_test_t.to(device))
    predict_torch = saidas_teste.argmax(dim=1).cpu().numpy()  # classe com maior pontuação

y_test_int = y_test.astype(int)
print(f"Acurácia da rede neural (PyTorch): {100 * (predict_torch == y_test_int).mean():.2f}%")
print(f"Acurácia da rede neural (scikit-learn): {100 * mlp.score(X_test, y_test):.2f}%")

### Calculando outras métricas (Precisão, F1-Score, Recall)

In [ ]:
print(f"Classification report (PyTorch):\n{metrics.classification_report(y_test_int, predict_torch)}")

In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_predictions(y_test_int, predict_torch)
disp.figure_.suptitle("Confusion Matrix - PyTorch")
plt.show()

### Visualizando algumas previsões

In [ ]:
probs = torch.softmax(saidas_teste, dim=1).cpu()          # converte as saídas em probabilidades

_, axes = plt.subplots(nrows=2, ncols=6, figsize=(12, 4.5))
for ax, i in zip(axes.ravel(), range(12)):
    cor = "green" if predict_torch[i] == y_test_int[i] else "red"  # verde = acerto, vermelho = erro
    ax.imshow(X_test[i].reshape(28, 28), cmap=plt.cm.gray_r)
    ax.set_title(f"Previsto: {predict_torch[i]} ({probs[i].max():.0%})\nReal: {y_test_int[i]}", color=cor, fontsize=9)
    ax.set_axis_off()
plt.tight_layout(); plt.show()

### Para experimentar

1. Aumente os neurônios das camadas ocultas (ex.: 128 → 64 → 32). O que acontece com a acurácia e com o número de parâmetros?
2. Troque o otimizador por `torch.optim.Adam(modelo.parameters(), lr=1e-3)`.
3. Treine por mais épocas e observe a curva de aprendizado.
4. No Colab, ative a GPU (*Ambiente de execução → Alterar o tipo de ambiente de execução*) e compare o tempo de treino.

**Documentação:** https://pytorch.org/docs/stable/nn.html e https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html